# Symbol alphabet and trellis

Top: every value the modulator can emit at $L_0=3$ with its multiplicity, against a Gaussian of the same mean and variance. Bottom: the symbols reachable in three steps from the all-zero state, coloured by the input pair.

In [ ]:
import subprocess, pathlib, collections
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = pathlib.Path.cwd().parent
MSPRS = ROOT / "build" / "bin" / "msprs"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 9, "axes.labelsize": 10,
    "legend.fontsize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.linewidth": 0.8, "figure.dpi": 110, "savefig.dpi": 200,
    "savefig.bbox": "tight", "pdf.fonttype": 42,
})

L0 = 3
STYLE = {(0, 0): ("#1f77b4", "o"), (0, 1): ("#d62728", "s"),
         (1, 0): ("#2ca02c", "^"), (1, 1): ("#ff7f0e", "D")}

def trellis(family):
    """(state, b0, b1) -> (symbol, next_state), straight from the simulator."""
    out = subprocess.run([str(MSPRS), "--mode", "alphabet", "--L0", str(L0),
                          "--family", family], capture_output=True, text=True,
                         check=True).stdout
    table = {}
    for line in out.splitlines():
        if line.startswith("#"):
            continue
        s, b0, b1, sym, nxt = line.split()
        table[(int(s), int(b0), int(b1))] = (float(sym), int(nxt))
    return table

In [ ]:
def walk(table, steps=3):
    """Every path of `steps` symbols out of state 0."""
    paths = [(0, [])]
    levels = []
    for _ in range(steps):
        nxt, edges = [], []
        for state, hist in paths:
            for b0 in (0, 1):
                for b1 in (0, 1):
                    sym, ns = table[(state, b0, b1)]
                    edges.append((hist[-1] if hist else None, sym, (b0, b1)))
                    nxt.append((ns, hist + [sym]))
        levels.append(edges)
        paths = nxt
    return levels

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.6), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2.6], "hspace": 0.08})

for col, family in enumerate(("balanced", "unbalanced")):
    table = trellis(family)
    syms = [v[0] for v in table.values()]
    counts = collections.Counter(round(s, 9) for s in syms)
    lv = np.array(sorted(counts))
    mult = np.array([counts[v] for v in lv])

    # ── multiplicity against a matching Gaussian ──
    ax = axes[0, col]
    mu = float(np.average(lv, weights=mult))
    var = float(np.average((lv - mu) ** 2, weights=mult))
    sd = np.sqrt(var)
    st = ax.stem(lv, mult, basefmt=" ")
    st.markerline.set(color="#808080", markersize=4)
    st.stemlines.set(color="#808080", linewidth=1.2)
    g = np.linspace(-3.2, 3.2, 400)
    pdf = np.exp(-0.5 * ((g - mu) / sd) ** 2)
    ax.plot(g, pdf * mult.max(), color="#000000", lw=1.6, zorder=1)
    ax.set(title=family.capitalize(), xlim=(-3.2, 3.2), yticks=[],
           ylim=(0, mult.max() * 1.45))
    ax.text(0.5, 0.99, rf"$\mu={mu:.3f},\ \sigma={sd:.3f}$", transform=ax.transAxes,
            ha="center", va="top", fontsize=9)
    for s in ("left", "right", "top"):
        ax.spines[s].set_visible(False)

    # ── the tree ──
    ax = axes[1, col]
    levels = walk(table)
    for step, edges in enumerate(levels, start=1):
        for prev, sym, pair in edges:
            colour, marker = STYLE[pair]
            y0 = step - 1
            x0 = prev if prev is not None else 0.0
            ax.annotate("", xy=(sym, step), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle="->", color=colour,
                                        lw=0.7, alpha=0.55))
            ax.plot([sym], [step], marker=marker, color=colour, ms=7,
                    markeredgecolor="#333333", markeredgewidth=0.5, zorder=3)
    ax.plot([0], [0], "o", color="#555555", ms=8, zorder=3)
    ax.plot(lv, np.full_like(lv, 3.75), "o", color="#000000", ms=6, zorder=3)
    ax.set(xlim=(-3.2, 3.2), ylim=(-0.4, 4.1),
           yticks=[0, 1, 2, 3, 3.75],
           yticklabels=["init", "step 1", "step 2", "step 3", "alphabet"],
           xlabel="Symbol $s[k]$")
    ax.grid(axis="x", alpha=0.3, ls=":")
    ax.axvline(0.0, color="0.6", ls="--", lw=0.8, zorder=0)

handles = [Line2D([], [], color=c, marker=m, ls="none", ms=8,
                  label=f"$(b_0, b_1) = {a}{b}$")
           for (a, b), (c, m) in STYLE.items()]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.99))
fig.subplots_adjust(top=0.90)
for ext in ("pdf", "png"):
    fig.savefig(FIGURES / f"constellations_overview.{ext}", facecolor="white")